This notebook prepares the data needed for the blending notebook in this folder.

The blending notebook has already loaded the RHMSS radar composite (`precip`, `metadata`) and the ECMWF IFS forecast for the case of 12 August 2017 (`nwp_precip_native`, `nwp_metadata_native`). Here we put the NWP forecast on the radar grid and on the nowcast time steps, and do the pre-processing that the blending methods need.

## The radar composite

The blending notebook has given us the composite as `precip` (in mm/h) and its `metadata`. Two things remain: `pysteps.utils.reproject_grids` assumes a north-up raster while the RHMSS composite is stored south-up, so we flip it; and we split it into nowcast input and observations.

The issue time is chosen with some care. A few composites in this sequence, among them the one at 15:50 UTC, are missing part of the radar network: about 56% of the domain is no-data instead of the usual 30%. If such a frame is used as nowcast input, the optical flow locks onto the moving edge of the coverage gap rather than onto the rain, and returns speeds of several hundred km/h. Issuing at 15:40 UTC keeps all four input composites clean.

In [ ]:
import datetime
import numpy as np
import pysteps

# Flip the radar composite to a north-up raster
radar_precip_full = precip[:, ::-1, :] if metadata["yorigin"] == "lower" else precip.copy()
radar_metadata = metadata.copy()
radar_metadata["yorigin"] = "upper"

timestep = int(radar_metadata["accutime"])   # 5 minutes
n_nowcast_steps = 24                         # 2 hours ahead
issue_index = 8                              # 15:40 UTC: the last issue time whose
                                             # four input composites are all complete

# Four composites as nowcast input, the rest as observations to verify against
radar_precip = radar_precip_full[issue_index - 3:issue_index + 1]
precip_obs = radar_precip_full[issue_index + 1:]
date_radar = radar_metadata["timestamps"][issue_index]

print(f"forecast issued at {date_radar:%Y-%m-%d %H:%M} UTC")
print(f"verifying against {len(precip_obs)} observations, "
      f"up to {radar_metadata['timestamps'][-1]:%H:%M} UTC")
# The 15:50 UTC composite is incomplete; it is not used as input, but it is still
# one of the observations we verify against (at +10 min).

## Onto the radar grid and the nowcast time steps

The NWP forecast is the ECMWF IFS control run initialised at 00 UTC, in mean rain rates. Two mismatches with the radar composite have to be resolved before we can blend.

**In space**, the IFS is on a regular lat/lon grid of about 10 km and the radar on a 1 km Lambert azimuthal equal-area grid, so we reproject the NWP onto the radar grid.

**In time**, the IFS is hourly while the nowcast runs every 5 minutes. Holding each hourly field constant would freeze the NWP component for an hour at a time, so we first interpolate it to 5-minute fields with the advection-based method of [block 3a](block_03a_advection_interpolation.ipynb), described in detail in [block 5a](block_05a_nwp_temporal_interpolation.ipynb), and then sample that at the nowcast steps. Interpolating straight to the 5-minute nowcast time step matters for more than smoothness: it means every lead time gets its own NWP field, so the optical flow between two consecutive NWP fields really is a displacement per nowcast time step.

In [ ]:
from scipy.ndimage import map_coordinates
from pysteps import motion
from pysteps.utils import transformation

# --- interpolate the hourly NWP to 15 minutes, on its own grid ---------------
# An hourly field is the mean rate over the preceding hour, so it represents the
# midpoint of that interval rather than its label time.
nwp_timestep = int(nwp_metadata_native["accutime"])
midpoints = np.array([t - datetime.timedelta(minutes=nwp_timestep / 2)
                      for t in nwp_metadata_native["timestamps"]])

horizon_end = date_radar + datetime.timedelta(minutes=timestep * n_nowcast_steps)
i0 = max(int(np.searchsorted(midpoints, date_radar, side="right")) - 1, 0)
i1 = min(int(np.searchsorted(midpoints, horizon_end, side="left")), len(midpoints) - 1)
seq, seq_times = nwp_precip_native[i0:i1 + 1], midpoints[i0:i1 + 1]

seq_dbr, _ = transformation.dB_transform(seq, nwp_metadata_native,
                                         threshold=0.1, zerovalue=-15.0)
nwp_motion = motion.get_method("LK")(seq_dbr, verbose=False)

interp_timestep = timestep   # 5 minutes: one NWP field per nowcast time step
n_sub = nwp_timestep // interp_timestep
gx, gy = np.meshgrid(np.arange(seq.shape[2], dtype=float),
                     np.arange(seq.shape[1], dtype=float))


def interpolate(precip1, precip2, motion_field):
    """Return the n_sub sub-interval mean rates between two NWP fields."""
    out = []
    for k in range(n_sub):
        f = (k + 0.5) / n_sub
        # Advect the first field forward and the second one backward, so that
        # both are valid at the same time, then average them
        coords1 = (gy - f * motion_field[1], gx - f * motion_field[0])
        coords2 = (gy + (1 - f) * motion_field[1], gx + (1 - f) * motion_field[0])
        out.append((1 - f) * map_coordinates(precip1, coords1, order=1, mode="nearest")
                   + f * map_coordinates(precip2, coords2, order=1, mode="nearest"))
    return out


nwp_fine, nwp_fine_times = [], []
for i in range(len(seq) - 1):
    nwp_fine.extend(interpolate(seq[i], seq[i + 1], nwp_motion))
    nwp_fine_times.extend(seq_times[i] + datetime.timedelta(minutes=interp_timestep * (k + 1))
                          for k in range(n_sub))
nwp_fine = np.stack(nwp_fine)

# --- reproject onto the radar grid -------------------------------------------
nwp_fine, nwp_metadata = pysteps.utils.reproject_grids(
    src_array=nwp_fine, dst_array=radar_precip[-1:],
    metadata_src=nwp_metadata_native, metadata_dst=radar_metadata)
nwp_fine = np.nan_to_num(nwp_fine)   # the IFS domain stops short in the east


# --- sample at the nowcast time steps ----------------------------------------
def resample_nwp_in_time(fields, times, valid_times):
    """Pick, for each requested time, the NWP interval that contains it."""
    idx = np.searchsorted(np.asarray(times), np.asarray(valid_times), side="left")
    return fields[np.clip(idx, 0, len(times) - 1)]


lead_times = [date_radar + datetime.timedelta(minutes=timestep * (i + 1))
              for i in range(n_nowcast_steps)]
nwp_precip = resample_nwp_in_time(nwp_fine, nwp_fine_times, lead_times)
# STEPS blending also needs the field at the issue time itself
nwp_precip_steps = resample_nwp_in_time(nwp_fine, nwp_fine_times, [date_radar] + lead_times)

print("NWP on the radar grid:", nwp_precip.shape,
      "|", len(np.unique([f.tobytes() for f in nwp_precip])), "distinct fields")

## Pre-processing for the nowcast

Threshold the data, transform the radar fields to dB (which makes the optical flow and the nowcasts better behaved) and estimate the motion field. The mm/h metadata is kept separate from the dB metadata, so that we can always convert back.

In [ ]:
# Simple threshold of 0.1 mm/h
radar_precip[radar_precip < 0.1] = 0.0
nwp_precip[nwp_precip < 0.1] = 0.0
nwp_precip_steps[nwp_precip_steps < 0.1] = 0.0

transformer = pysteps.utils.get_method("dB")
radar_precip_db, radar_metadata_db = transformer(radar_precip, radar_metadata, threshold=0.1)

oflow_method = pysteps.motion.get_method("lucaskanade")
velocity_radar = oflow_method(radar_precip_db)
print("motion field:", velocity_radar.shape)